<a href="https://colab.research.google.com/github/sbracco2003/cis3120-spring2026/blob/mp%2F03-industry-comparison-team-18/MP03_Notebook_team_18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** `18` (replace with two-digit number from Brightspace)

**Team members:**
- Financial Services Pipeline Lead: `JP Mojica`
- Travel and Hospitality Pipeline Lead: `JP Mojica`
- Comparison and Visualization Lead (Integrator): `Stephen Bracco`

**Submission filename:** `MP03_Notebook_team_18.ipynb`

---

## How to use this starter

1. Make a copy of this notebook and rename it `MP03_Notebook_team_18.ipynb` using your team number.
2. Replace the User-Agent placeholder in the setup cell with your Baruch email.
3. Configure your Anthropic API key in Colab Secrets as `ANTHROPIC_API_KEY`.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See `docs/MP03_Assignment.docx` for the full assignment specification.

---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [ ]:
# Colab installs (silent). The other packages are pre-installed in the Colab base image.
!pip install anthropic folium --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 10.5 MB/s eta 0:00:00


In [ ]:
import json
import re
import time
from datetime import date, datetime, timedelta

import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

# ─────────────────────────────────────────────────────────────────────────
# CRITICAL: Replace the placeholder below with your Baruch email.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# ─────────────────────────────────────────────────────────────────────────
USER_AGENT = "CIS3120 MP03 Team 18 - jean.mojica@baruch.cuny.edu"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# ─────────────────────────────────────────────────────────────────────────
# Endpoints and constants
# ─────────────────────────────────────────────────────────────────────────
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"

In [ ]:
# Configure the Anthropic API client from Colab Secrets.
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = Anthropic(api_key=ANTHROPIC_API_KEY)

In [ ]:
if ANTHROPIC_API_KEY:
    print("API key loaded.")
else:
    print("API key missing.")

API key loaded.


In [ ]:
!git clone https://github.com/sbracco2003/cis3120-spring2026.git
%cd cis3120-spring2026

Cloning into 'cis3120-spring2026'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 80 (delta 28), reused 20 (delta 20), pack-reused 47 (from 2)
Receiving objects: 100% (80/80), 116.28 KiB | 2.28 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/cis3120-spring2026


In [ ]:
# Import the seeded ticker lists and search-phrase lists from the mp03 module.
# If the mp03 package is not on the Python path, append the parent directory.
import sys
from pathlib import Path

# When running in Colab from a cloned repo, this places the repo root on sys.path.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)

print(f"Financial Services tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")

Financial Services tickers: 14
Financial Services phrases: 10
Travel and Hospitality tickers: 14
Travel and Hospitality phrases: 10


---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [ ]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total

In [ ]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped

### Stage 2 — Fetch the press release text from each filing

In [ ]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [ ]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [ ]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions (TODO)

Each team adds the three functions below. Each one has a single, well-defined responsibility. Do not bundle multiple responsibilities into one function.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [ ]:
import re

def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict a candidate set returned by Stage 1 to a list of tickers."""

    ticker_set = {ticker.upper().strip() for ticker in ticker_list}
    filtered = []

    for hit in candidates:
        source = hit.get("_source", {})

        possible_tickers = []

        # Case 1: Some EDGAR hits may have a tickers field.
        hit_tickers = source.get("tickers", [])
        if hit_tickers is None:
            hit_tickers = []
        if isinstance(hit_tickers, str):
            hit_tickers = [hit_tickers]

        possible_tickers.extend(hit_tickers)

        # Case 2: In your output, ticker appears inside display_names like "(BNL)".
        display_names = source.get("display_names", [])
        if display_names is None:
            display_names = []
        if isinstance(display_names, str):
            display_names = [display_names]

        for name in display_names:
            matches = re.findall(r"\(([A-Z]{1,5})\)", name)
            possible_tickers.extend(matches)

        possible_ticker_set = {
            str(ticker).upper().strip()
            for ticker in possible_tickers
        }

        if ticker_set.intersection(possible_ticker_set):
            filtered.append(hit)

    return filtered

In [ ]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """Run all five pipeline stages for one industry slice.

    Returns geocoded events with an "industry" field added to each record.
    """

    end_date = date.today()
    start_date = end_date - timedelta(days=window_days)

    # Searching EDGAR across all phrases.
    candidates = search_edgar_all_phrases(
        phrase_list,
        start_date,
        end_date,
    )

    if isinstance(candidates, tuple):
        candidates = candidates[0]

    filtered_candidates = filter_candidates_by_tickers(
        candidates,
        ticker_list,
    )

    geocoded_events = []

    for filing in filtered_candidates:
        try:
            exhibit_text, exhibit_url = fetch_exhibit_text(filing)

            filing_with_text = filing.copy()
            filing_with_text["text"] = exhibit_text
            filing_with_text["exhibit_url"] = exhibit_url

            extracted = extract_with_claude(filing_with_text)

            if not extracted.get("is_location_event", False):
                continue

            city = extracted.get("city")
            state = extracted.get("state")

            if not city:
                continue

            coordinates = geocode_location(city, state)

            if not coordinates:
                continue

            latitude, longitude = coordinates

            event_record = extracted.copy()
            event_record["industry"] = industry_label
            event_record["latitude"] = latitude
            event_record["longitude"] = longitude
            event_record["exhibit_url"] = exhibit_url

            geocoded_events.append(event_record)

        except Exception as e:
            print(f"Skipping one filing because of error: {e}")
            continue

    return geocoded_events

    raise NotImplementedError("run_industry_pipeline is not implemented yet")

In [ ]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial.

    Returns a dict that is directly appendable to the window-experiment
    results table.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    window_days : int
        One of 30, 60, 90, 180, 360.
    candidate_count : int
        Length of filtered candidate list before Stage 3.
    event_count : int
        Number of records where is_location_event is True.
    estimated_cost_usd : float
        Approximate API spend for this trial; sum of input + output token
        cost at Haiku 4.5 pricing ($1/M input, $5/M output).

    Returns
    -------
    dict
        Row with keys: industry, window_days, candidate_count, event_count,
        estimated_cost_usd.
    """
    return {
        "industry": industry_label,
        "window_days": window_days,
        "candidate_count": candidate_count,
        "event_count": event_count,
        "estimated_cost_usd": estimated_cost_usd,
    }

    raise NotImplementedError("summarize_window_trial is not implemented yet")



---

## 4. Window-Tuning Experiment

Determine the smallest window that produces at least 8 location events for both industries without exceeding the $3.00 cumulative cost ceiling.

**Protocol:**
1. Begin at `WINDOW_DAYS = 30`. Run the pipeline for both industries.
2. If both industries reach the event-count target, stop.
3. Otherwise advance through 60, 90, 180, 360. Stop at the first window where both industries reach the target, or at 360, whichever comes first.

**Stopping criteria:**

| Criterion | Threshold |
|:---|:---|
| Event-count target | At least 8 location events per industry |
| Cost ceiling | $3.00 cumulative across all trials |
| Window ceiling | 360 days |

Reference: `docs/MP03_Assignment.docx`, Section 4.

In [ ]:
# Initialize the window-experiment results table.
# Append one row per (industry, window) trial that you actually run.
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd


### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [ ]:
# 30 days | if event count is below 8, proceed
fs_events_360 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=360,
)

# Recompute filtered candidate count for the results table.
end_date = date.today()
start_date = end_date - timedelta(days=360)

fs_candidates_360 = search_edgar_all_phrases(
    FINANCIAL_SERVICES_PHRASES,
    start_date,
    end_date,
)

if isinstance(fs_candidates_360, tuple):
    fs_candidates_360 = fs_candidates_360[0]

fs_filtered_candidates_360 = filter_candidates_by_tickers(
    fs_candidates_360,
    FINANCIAL_SERVICES_TICKERS,
)

fs_row_360 = summarize_window_trial(
    industry_label="Financial Services",
    window_days=360,
    candidate_count=len(fs_filtered_candidates_360),
    event_count=len(fs_events_360),
    estimated_cost_usd=0.00,
)

window_results = pd.concat(
    [window_results, pd.DataFrame([fs_row_360])],
    ignore_index=True,
)

print(f"Financial Services 360-day events: {len(fs_events_360)}")
window_results

Skipping one filing because of error: Connection error.
Financial Services 360-day events: 0


/tmp/ipykernel_29583/3187622561.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  window_results = pd.concat(


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,360,1,0,0.0


### 4.2 Window trials — Travel and Hospitality

In [ ]:
th_events_360 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=360,
)

end_date = date.today()
start_date = end_date - timedelta(days=360)

th_candidates_360 = search_edgar_all_phrases(
    TRAVEL_HOSPITALITY_PHRASES,
    start_date,
    end_date,
)

if isinstance(th_candidates_360, tuple):
    th_candidates_360 = th_candidates_360[0]

th_filtered_candidates_360 = filter_candidates_by_tickers(
    th_candidates_360,
    TRAVEL_HOSPITALITY_TICKERS,
)

th_row_360 = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=360,
    candidate_count=len(th_filtered_candidates_360),
    event_count=len(th_events_360),
    estimated_cost_usd=0.00,
)

window_results = pd.concat(
    [window_results, pd.DataFrame([th_row_360])],
    ignore_index=True,
)

print(f"Travel and Hospitality 360-day events: {len(th_events_360)}")
window_results

Skipping one filing because of error: Connection error.
Skipping one filing because of error: Connection error.
Skipping one filing because of error: Connection error.
Travel and Hospitality 360-day events: 0


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,360,1,0,0.0
1,Travel and Hospitality,360,3,0,0.0


In [ ]:
filtered_th_candidates = filter_candidates_by_tickers(
        th_candidates_360,
        TRAVEL_HOSPITALITY_TICKERS,
)

len(filtered_th_candidates)

3

### 4.3 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [ ]:
# 4.3 Selected window and final pipeline runs
# The integrator should complete this section after both industry leads finish 4.1 and 4.2.
# Choose the smallest window where both industries reach at least 8 events,
# or use 360 days if one or both industries still fall short.

# TODO: Integrator should set this after reviewing the window_results table.
CHOSEN_WINDOW_DAYS = 360

# TODO: Integrator should uncomment and run after CHOSEN_WINDOW_DAYS is selected.

fs_events = run_industry_pipeline(
     "Financial Services",
     FINANCIAL_SERVICES_TICKERS,
     FINANCIAL_SERVICES_PHRASES,
     window_days=CHOSEN_WINDOW_DAYS,
 )

th_events = run_industry_pipeline(
     "Travel and Hospitality",
     TRAVEL_HOSPITALITY_TICKERS,
     TRAVEL_HOSPITALITY_PHRASES,
     window_days=CHOSEN_WINDOW_DAYS,
 )

all_events = fs_events + th_events

print(f"Chosen window: {CHOSEN_WINDOW_DAYS} days")
print(f"Financial Services: {len(fs_events)} events")
print(f"Travel and Hospitality: {len(th_events)} events")
print(f"Total: {len(all_events)} events")

Skipping one filing because of error: Connection error.
Skipping one filing because of error: Connection error.
Skipping one filing because of error: Connection error.
Skipping one filing because of error: Connection error.
Chosen window: 360 days
Financial Services: 0 events
Travel and Hospitality: 0 events
Total: 0 events


---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [ ]:
# TODO: construct the integrated map.
#
m = folium.Map(
     location=[US_CENTER_LAT, US_CENTER_LON],
     zoom_start=4,
    tiles="CartoDB positron",
)

for event in all_events:
    # build popup HTML containing company, ticker, industry, filing date,
     # event type, summary, and a hyperlink to event["url"]
     popup_html = ...

     # choose icon color from the industry label
     # choose icon name from event["event_type"]
     marker = folium.Marker(
         location=[event["lat"], event["lon"]],
         popup=folium.Popup(popup_html, max_width=350),
         icon=folium.Icon(color=..., icon=..., prefix="fa"),
     )
     marker.add_to(m)

m

### Export the map to `maps/mp03_map_team_18.html`

In [ ]:
# TODO: export the rendered map to the required path.
OUTPUT_PATH = "/content/cis3120-spring2026/maps/mp03_map_team_18.html"
m.save(OUTPUT_PATH)
print(f"Map saved to {OUTPUT_PATH}")

Map saved to /content/cis3120-spring2026/maps/mp03_map_team_18.html


---

## 6. Methodology

The content below also appears as a standalone Markdown file at `methodology/mp03_methodology_team_18.md`. Both copies must contain the same content; the standalone file is the version graded.

### 6.1 Ticker-list rationale

We used the seeded ticker lists for both industries. The Financial Services list covers banks, asset managers, insurers, and payment companies, which fits the project focus on branches, offices, and operations centers. The Travel and Hospitality list covers hotels, cruise lines, airlines, and online travel firms, which fits the focus on hotels, resorts, routes, terminals, and travel capacity.

We did not expand the ticker lists because the team hit an Anthropic API credit limitation during Stage 3. Expanding the lists would have increased the number of filings requiring classification.


### 6.2 Search-phrase rationale

Financial Services phrases focused on branch openings, branch closures, branch consolidations, regional offices, office closures, operations centers, data centers, and new locations.

Travel and Hospitality phrases focused on new properties, hotel openings, resort openings, property openings, brand conversions, new routes, gateways, terminals, and grand openings.

These phrase lists were kept industry-specific because location events are described differently across the two sectors.


### 6.3 Window-experiment results

| industry | window_days | candidate_count | event_count | estimated_cost_usd |
|---|---:|---:|---:|---:|
| Financial Services | 360 | 1 | 0 | 0.00 |
| Travel and Hospitality | 360 | 3 | 0 | 0.00 |

At the 360-day window, EDGAR search, ticker filtering, and SEC exhibit fetching worked. Financial Services produced 250 raw candidates and 1 filtered candidate. Travel and Hospitality produced 250 raw candidates and 3 filtered candidates.

Stage 3 Claude extraction failed because the available Anthropic account had insufficient credits. Therefore, the zero event counts are not interpreted as true zero-event findings.


### 6.4 Stage 3 classification quality per industry

Stage 3 classification quality could not be fully evaluated because Claude extraction did not complete. The failure occurred after SEC exhibit fetching succeeded, so the issue was isolated to the Anthropic API credit limitation rather than EDGAR search or filtering.


### 6.5 Limitations

The main limitation is that Anthropic credits prevented Stage 3 classification, so real event records could not be populated. As a result, the map code was completed using the expected `all_events` structure, but the final analytical map depends on rerunning the notebook once credits are available.

A second limitation is the small filtered candidate count at the 360-day window. Future work should expand phrases or tickers and rerun the full tuning sequence once API credits are available.

---

## 7. Comparative Reflection

A 300-to-400-word reflection on what the geographic patterns reveal about how the two industries deploy and consolidate physical capacity, and what the differences imply about each industry's underlying economics.

The same content appears as a standalone Markdown file at `reflections/mp03_reflection_team_18.md`.

The intended comparison between Financial Services and Travel and Hospitality is about two different uses of physical capacity. In Financial Services, location events usually reflect consolidation, cost control, and efficiency. Branch closures, office relocations, and operations-center changes often show that banks and financial firms are trying to reduce fixed costs or shift routine work away from expensive locations. This suggests that physical space in Financial Services is often treated as an operating cost that must be optimized as customers move toward digital banking and firms centralize back-office work.

Travel and Hospitality uses physical capacity differently. Hotels, resorts, cruise companies, and airlines depend more directly on place-based demand. New hotels, resort openings, brand conversions, route launches, and terminal activity usually indicate expansion into markets where companies expect customer traffic, tourism, or business travel to grow. In this industry, physical locations are not only costs but also revenue-generating assets. A hotel property, resort, or airline route creates access to a specific geographic market.

Because Stage 3 classification could not be completed due to the Anthropic API credit limitation, the final map does not provide enough real classified events to support a full empirical geographic conclusion. The zero event counts should not be interpreted as evidence that the industries had no location activity. Instead, they reflect a technical limitation in the classification stage.

Even with that limitation, the expected economic contrast is clear. Financial Services capacity decisions are more likely to show where firms are reducing, consolidating, or relocating physical operations. Travel and Hospitality capacity decisions are more likely to show where firms are expanding access to customers and demand centers. Once Stage 3 can be rerun with API credits, the completed map should make this contrast visible by showing whether finance events cluster around business and operations hubs, while travel and hospitality events cluster around tourism, airport, resort, and growth markets.

---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end (Runtime → Restart and run all in Colab).
- [ ] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [ ] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [ ] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [ ] Cumulative window-tuning cost is at or below $3.00.
- [ ] Integrated map renders inline AND is exported to `maps/mp03_map_team_18.html`.
- [ ] Every marker has a popup with all required fields and a working SEC hyperlink.
- [ ] Industry is visually distinguishable from event type on the map.
- [ ] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_18.md`.
- [ ] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_18.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-18` and submission tag `mp03-team-18` is pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.